# Waveform Feature Dataset (all sessions, all units)

Build **one big table** that contains, for **every unit of every session**, its
peak-channel waveform, morphology features, brain region, and a **QC label**
(passed default QC or not). Save it once to a single CSV so downstream analysis
(feature exploration, clustering, ...) can just reload the CSV instead of
re-reading the NWBs every time.

Each row contains:

- `session_name` — session-core id
- `unit_index` — within-session unit index
- `region` — CCF brain region (or empty)
- `qc_pass` — `True` if the unit passed default QC, else `False`
- the morphology features (`trough_to_peak_ms`, `half_width_ms`, ...)
- `pre_ms` / `post_ms` / `sampling_rate_hz` — timing context (to rebuild the axis)
- `wf_000 ... wf_NNN` — the raw trough-aligned peak-channel waveform samples

All the reusable logic lives in `waveform_clustering.py`
(`build_features_dataset`, `save_dataset_csv`, `load_features_dataset`), so this
notebook stays thin.

**Workflow**
1. Discover all sessions and build the full per-unit dataset.
2. Save it to a single CSV.
3. **Example** — reload the CSV and run the same feature / clustering /
   visualization workflow as `waveform_clustering.ipynb`.


## 1. Imports & configuration

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP & MODULE IMPORTS
# =============================================================================
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Path to the custom analysis modules (update for your local install).
MODULE_PATH = Path("/root/capsule/src/aind_dft_ephys_analysis")
if str(MODULE_PATH) not in sys.path:
    sys.path.insert(0, str(MODULE_PATH))

print(f"Analysis modules loaded from: {MODULE_PATH}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.decomposition import PCA

# Project utilities (src/aind_dft_ephys_analysis)
from general_utils import find_ephys_sessions

# Reusable waveform-clustering functions (see waveform_clustering.py)
from waveform_clustering import (
    FEATURE_COLS,
    WAVEFORM_COL_PREFIX,
    build_features_dataset,
    save_dataset_csv,
    load_features_dataset,
    normalize_waveform,
    scale_features,
    estimate_n_clusters,
    cluster_waveforms,
)

%matplotlib inline

# ---- Configuration -------------------------------------------------------
# Waveform sampling rate (Neuropixels standard). Adjust if your probe differs.
SAMPLING_RATE_HZ = 30_000.0

# Fixed window (ms) extracted around each unit's trough so waveforms from
# probes/sessions with different sample counts can be pooled together.
PRE_MS = 1.0
POST_MS = 2.0

# Where to write / read the big dataset CSV.
RESULTS_DIR = Path("/root/capsule/scratch/waveform_clustering")
DATASET_CSV = RESULTS_DIR / "all_sessions_waveform_features.csv"

RANDOM_STATE = 0
feature_cols = FEATURE_COLS

## 2. Build the full per-unit dataset

Loop over every (spike-sorted) session and, for **every** unit that yields a
valid trough-aligned waveform, extract its peak-channel waveform + morphology
features and label whether it passed default QC. This reads each NWB once.

> This step can take a while (it opens every session). Once saved (next
> section) you can skip straight to the reload/example sections.


In [ ]:
# Discover all ephys sessions (prefer spike-sorted ones).
all_sessions, sessions_by_animal, spike_sorted_sessions = find_ephys_sessions()
sessions_to_use = spike_sorted_sessions if len(spike_sorted_sessions) else all_sessions
print(f"Found {len(all_sessions)} sessions; using {len(sessions_to_use)} spike-sorted sessions.")

# Build the big table: every unit of every session, QC-labelled, with the raw
# trough-aligned waveform samples (wf_000 ... wf_NNN) stored inline.
dataset = build_features_dataset(
    sessions_to_use,
    pre_ms=PRE_MS,
    post_ms=POST_MS,
    sampling_rate_hz=SAMPLING_RATE_HZ,
    qc_metric_fallback=True,
    normalize=True,          # features computed from normalized waveforms
    include_waveform=True,   # store the per-sample waveform columns
    verbose=True,
)

print(f"\nDataset shape: {dataset.shape[0]} units x {dataset.shape[1]} columns")
dataset.head()

In [ ]:
# Quick overview: QC pass/fail counts, per-region and per-session tallies.
print("QC label counts:")
print(dataset["qc_pass"].value_counts(dropna=False), "\n")

print("Units per region (top 20):")
print(dataset["region"].fillna("None").value_counts().head(20), "\n")

print(f"Sessions represented: {dataset['session_name'].nunique()}")
print(dataset["session_name"].value_counts().head(10))

## 3. Save the dataset to a single CSV

Write everything (metadata + QC label + features + waveform samples) to one
CSV. Reload it any time instead of re-reading the NWBs.


In [ ]:
csv_path = save_dataset_csv(dataset, out_dir=RESULTS_DIR, filename=DATASET_CSV.name)
print(f"Saved {len(dataset)} units to: {csv_path}")

## 4. Example — reload the CSV and analyze

Everything below only needs the saved CSV. `load_features_dataset` splits the
flat table back into a feature/metadata DataFrame plus a waveform matrix, and
rebuilds the trough-aligned time axis from the stored timing columns.


In [ ]:
loaded = load_features_dataset(DATASET_CSV)
features = loaded["features"]      # metadata + QC label + features
waveforms = loaded["waveforms"]    # (n_units, win_len) raw trough-aligned traces
time_ms = loaded["time_ms"]

print(f"Reloaded {len(features)} units; waveform matrix {waveforms.shape}.")
print(f"Time axis: {time_ms[0]:.2f} .. {time_ms[-1]:.2f} ms ({len(time_ms)} samples)")
features.head()

### 4a. Inspect the waveforms, split by QC label

Amplitude-normalize each stored (baseline-corrected, trough-aligned) waveform
and overlay QC-pass vs. QC-fail units.


In [ ]:
norm_waveforms = (
    np.vstack([normalize_waveform(w) for w in waveforms]) if len(waveforms) else waveforms
)
qc_pass = features["qc_pass"].astype(bool).values

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)
for ax, keep, title, color in (
    (axes[0], qc_pass, "QC pass", "C0"),
    (axes[1], ~qc_pass, "QC fail", "C3"),
):
    sub = norm_waveforms[keep]
    for w in sub:
        ax.plot(time_ms, w, color="lightgrey", linewidth=0.4)
    if len(sub):
        ax.plot(time_ms, sub.mean(axis=0), color=color, linewidth=2, label="mean")
    ax.set_title(f"{title} (n={int(keep.sum())})")
    ax.set_xlabel("Time (ms)")
    ax.legend()
axes[0].set_ylabel("Normalized amplitude")
plt.tight_layout()
plt.show()

### 4b. Choose the units to cluster

The morphology features are already in the reloaded table, so no recomputation
is needed. Here we cluster the **QC-passing** units (flip `USE_QC_ONLY` to use
all units).


In [ ]:
USE_QC_ONLY = True

if USE_QC_ONLY:
    mask = features["qc_pass"].astype(bool).values
else:
    mask = np.ones(len(features), dtype=bool)

feat = features.loc[mask].reset_index(drop=True).copy()
wf = norm_waveforms[mask]
print(f"Clustering {len(feat)} units (USE_QC_ONLY={USE_QC_ONLY}).")

# Distribution of the classic separator: trough-to-peak duration.
plt.figure(figsize=(7, 4))
plt.hist(feat["trough_to_peak_ms"], bins=30, color="C0", alpha=0.8)
plt.xlabel("Trough-to-peak duration (ms)")
plt.ylabel("Unit count")
plt.title("Trough-to-peak duration distribution")
plt.tight_layout()
plt.show()

### 4c. Standardize features and estimate the number of clusters

In [ ]:
X_scaled, _ = scale_features(feat, feature_cols)
diag = estimate_n_clusters(X_scaled, k_range=range(2, 8), random_state=RANDOM_STATE)
ks, inertias, silhouettes, best_k = (
    diag["ks"], diag["inertias"], diag["silhouettes"], diag["best_k"]
)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(ks, inertias, "o-")
axes[0].set_xlabel("k"); axes[0].set_ylabel("Inertia"); axes[0].set_title("Elbow")
axes[1].plot(ks, silhouettes, "o-", color="C1")
axes[1].set_xlabel("k"); axes[1].set_ylabel("Silhouette"); axes[1].set_title("Silhouette score")
plt.tight_layout()
plt.show()

print(f"Best k by silhouette: {best_k}")

### 4d. Cluster the units (KMeans + Gaussian Mixture)

In [ ]:
N_CLUSTERS = max(2, best_k)  # override manually if desired

clustering = cluster_waveforms(
    feat, n_clusters=N_CLUSTERS, feature_cols=feature_cols, random_state=RANDOM_STATE
)
X_scaled = clustering["X_scaled"]

labels = feat["cluster_gmm"].values  # switch to 'cluster_kmeans' if preferred
print(f"KMeans cluster sizes: {np.bincount(feat['cluster_kmeans'].values)}")
print(f"GMM cluster sizes:    {np.bincount(feat['cluster_gmm'].values)}")

### 4e. Visualize the clusters

In [ ]:
# Mean waveform per cluster
plt.figure(figsize=(8, 5))
for c in sorted(np.unique(labels)):
    m = labels == c
    plt.plot(time_ms, wf[m].mean(axis=0), linewidth=2, label=f"Cluster {c} (n={m.sum()})")
    for w in wf[m]:
        plt.plot(time_ms, w, color=f"C{c}", alpha=0.06, linewidth=0.5)
plt.xlabel("Time (ms)")
plt.ylabel("Normalized amplitude")
plt.title("Mean waveform per cluster")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Feature space: trough-to-peak vs half-width, colored by cluster
plt.figure(figsize=(7, 6))
sc = plt.scatter(feat["trough_to_peak_ms"], feat["half_width_ms"],
                 c=labels, cmap="tab10", s=25, alpha=0.8)
plt.xlabel("Trough-to-peak duration (ms)")
plt.ylabel("Half-width (ms)")
plt.title("Waveform feature space by cluster")
plt.colorbar(sc, label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# PCA projection of the standardized feature space
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(7, 6))
sc = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap="tab10", s=25, alpha=0.8)
plt.xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)")
plt.ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)")
plt.title("PCA of waveform features by cluster")
plt.colorbar(sc, label="Cluster")
plt.tight_layout()
plt.show()

In [ ]:
# Cluster composition by brain region, and mean feature values per cluster
composition = pd.crosstab(feat["region"].fillna("None"), feat["cluster_kmeans"])
print("Cluster composition per region:")
print(composition, "\n")

summary = feat.groupby("cluster_kmeans")[feature_cols].mean()
summary["n_units"] = feat.groupby("cluster_kmeans").size()
summary

## 5. Notes

- The big CSV round-trips the raw trough-aligned waveform via the `wf_###`
  columns, so `load_features_dataset` reconstructs both the features and the
  waveforms without touching the NWBs.
- `qc_pass` labels every unit, so you can cluster QC-passing units only, all
  units, or compare the two populations.
- Reusable helpers live in `waveform_clustering.py`
  (`build_features_dataset`, `save_dataset_csv`, `load_features_dataset`).
